# Phase 3 — Kaggle Inference (vLLM Ensemble + Majority Vote)

Runs all 3 fine-tuned SLMs sequentially and combines their predictions using
**character-level majority voting** to produce the final submission.

| Model | Size | Merge strategy |
|-------|------|---------------|
| `Qwen/Qwen3.5-9B` | 9B | 4-bit NF4 (disk: ~5 GB, VRAM: ~5 GB) |
| `google/gemma-4-E4B-it` | 4B | fp32 CPU merge (disk: ~8 GB, VRAM: ~8 GB) |
| `Qwen/Qwen3.5-4B` | 4B | fp32 CPU merge (disk: ~8 GB, VRAM: ~8 GB) |

**Default paths are set for Kaggle** (internet OFF).

| | |
|---|---|
| **Hardware** | T4 x2 on Kaggle |
| **Runtime** | ~30–60 min |
| **Output** | `/kaggle/working/submission.csv` |

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args], stdout=subprocess.DEVNULL)

# Install/upgrade all required packages
# vLLM 0.9.x supports Qwen3.5 and Gemma4 as text-only models (no VL handler routing)
pip("install", "-q", "--upgrade",
    "vllm>=0.9.0",
    "transformers>=5.5.0",
    "peft>=0.15.0",
    "bitsandbytes>=0.45.0",
    "rapidfuzz>=3.0.0",
    "xgrammar",
    "accelerate>=1.0.0",
)

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  {props.total_memory / 1024**3:.1f} GB")

import vllm, transformers, peft
print(f"vllm: {vllm.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")

## Configuration

> **Kaggle users**: paths below match default Kaggle dataset mount points.
>
> **Colab users**: change `DATA_DIR`, `ADAPTER_DIR`, and `OUTPUT_DIR` to local paths.

Key inference settings:

| Key | Default | Description |
|-----|---------|-------------|
| `ENFORCE_EAGER` | `True` | Required for clean VRAM release between models |
| `GPU_MEM_UTIL` | `0.85` | Lower to `0.80` if OOM |
| `VOTE_THRESHOLD` | `2` | Accept a character span if ≥ 2/3 models agree |

In [ ]:
import contextlib, gc, json, logging, re, shutil, sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from rapidfuzz.fuzz import partial_ratio_alignment
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM, AutoModelForImageTextToText, AutoTokenizer,
    BitsAndBytesConfig,
)
from vllm import LLM, SamplingParams
try:
    from vllm.sampling_params import GuidedDecodingParams
except ImportError:
    GuidedDecodingParams = None
    print("WARNING: GuidedDecodingParams not found — ensure vllm>=0.9.0 is installed")

try:
    from vllm.distributed.parallel_state import destroy_model_parallel
except ImportError:
    def destroy_model_parallel(): pass

CONFIG = {
    "DATA_DIR":                Path("/kaggle/input/nbme-score-clinical-patient-notes"),
    "ADAPTER_DIR":             Path("/kaggle/input/my-adapters"),
    "OUTPUT_DIR":              Path("/kaggle/working"),
    "ENFORCE_EAGER":           True,
    "GPU_MEM_UTIL":            0.85,
    "MAX_MODEL_LEN":           1024,
    "MAX_NEW_TOKENS":          1024,
    "LLM_TEMPERATURE":         0.0,
    "MAX_SPANS_PER_FEATURE":   10,
    "VOTE_THRESHOLD":          2,
    "FUZZY_SCORE_CUTOFF":      70.0,
    "SEED":                    42,
    "LARGE_MODEL_THRESHOLD_B": 7,
    "USE_VLLM":                True,   # False = transformers fallback (if vLLM routing fails)
}

MODEL_REGISTRY = [
    {
        "name":         "qwen_35_9b",
        "model_id":     "Qwen/Qwen3.5-9B",
        "model_class":  "causal_lm",
        "dtype":        torch.bfloat16,
        "vllm_dtype":   "bfloat16",
        "adapter_path": Path("/kaggle/input/my-adapters/qwen_35_9b_adapter"),
        "param_count":  9,
    },
    {
        "name":         "gemma_4_e4b",
        "model_id":     "google/gemma-4-E4B-it",
        "model_class":  "image_text_to_text",
        "dtype":        torch.bfloat16,
        "vllm_dtype":   "bfloat16",
        "adapter_path": Path("/kaggle/input/my-adapters/gemma_4_e4b_adapter"),
        "param_count":  4,
    },
    {
        "name":         "qwen_35_4b",
        "model_id":     "Qwen/Qwen3.5-4B",
        "model_class":  "causal_lm",
        "dtype":        torch.bfloat16,
        "vllm_dtype":   "bfloat16",
        "adapter_path": Path("/kaggle/input/my-adapters/qwen_35_4b_adapter"),
        "param_count":  4,
    },
]

SYSTEM_PROMPT = (
    "You are a clinical NLP specialist. "
    "Given a patient note and a clinical feature, extract the EXACT verbatim text spans "
    "from the note that express that feature. "
    "Rules:\n"
    "  1. Copy text character-for-character — do NOT paraphrase.\n"
    "  2. If the feature is absent from the note, return an empty list.\n"
    "  3. Output ONLY valid JSON — no markdown, no explanation.\n"
    'Output format: {"spans": ["exact text 1", "exact text 2"]}'
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

print("✓ CONFIG, MODEL_REGISTRY loaded")
print(f"  USE_VLLM = {CONFIG['USE_VLLM']}")

## Section 1 — Per-Note Regex FSM Constraint

For each patient note, we build a regex that only allows characters present in that note.
This is passed to vLLM's XGrammar backend as a guided decoding constraint — the model
**physically cannot hallucinate** characters that don't appear in the source note.

The regex enforces both:
1. **Structural**: output must be valid `{"spans": [...]}` JSON
2. **Lexical**: each span string can only contain characters from the note

In [ ]:
def _build_char_class(note_chars: set) -> str:
    """Convert a set of characters to a regex character class, escaping special chars."""
    parts = []
    for ch in sorted(note_chars, key=ord):
        code = ord(ch)
        if code < 0x20 or code == 0x7F:
            continue
        if ch == ']':   parts.append(r'\]')
        elif ch == '^': parts.append(r'\^')
        elif ch == '-': parts.append(r'\-')
        elif ch == '\\': parts.append(r'\\')
        else:           parts.append(ch)
    return '[' + ''.join(parts) + ']' if parts else r'[^\n]'


def build_constraint_regex(pn_history: str, max_spans: int = 10) -> str:
    """Build a per-note JSON regex that constrains span content to note characters."""
    note_chars = set(pn_history) - {'"', '\\'}
    char_class = _build_char_class(note_chars)
    span_item  = f'"{char_class}*"'
    additional = r'(?:, ' + span_item + r'){0,' + str(max_spans - 1) + r'}'
    opt_list   = r'(?:' + span_item + additional + r')?'
    pattern    = r'\{"spans": \[' + opt_list + r'\]\}'
    return pattern

print("✓ Section 1: _build_char_class, build_constraint_regex defined")

## Section 2 — LoRA Adapter Merger

Merges each LoRA adapter into the base model weights, then saves the merged weights
to a temp directory. vLLM then loads from the merged checkpoint.

**Merge strategy by model size:**
- **≥ 7B params (Qwen3.5-9B)**: load in 4-bit NF4 (requires GPU), merge, save quantized.
  9B fp16 = 18 GB disk + 18 GB VRAM → OOM on T4. 9B 4-bit = ~5 GB disk + ~5 GB VRAM → fits.
- **< 7B params (4B models)**: load on CPU, merge, save in fp32. No VRAM used during merge.

Benefits:
- Avoids vLLM's dynamic LoRA buffers (saves ~1 GB VRAM)
- Eliminates output drift from vLLM's LoRA multiplexing (issue #5148)
- Merged files are deleted after inference to reclaim disk space

In [ ]:
def merge_adapter_to_disk(model_spec: dict, output_dir: Path, cfg: dict) -> Path:
    model_id     = model_spec["model_id"]
    model_class  = model_spec["model_class"]
    adapter_path = model_spec["adapter_path"]
    dtype        = model_spec["dtype"]
    merged_path  = output_dir / f"merged_{model_spec['name']}"
    is_large     = model_spec.get("param_count", 0) >= cfg.get("LARGE_MODEL_THRESHOLD_B", 7)

    if merged_path.exists() and (merged_path / "config.json").exists():
        log.info(f"  [{model_spec['name']}] Merged model already on disk → {merged_path}")
        return merged_path

    # Pre-patch model_type via AutoConfig so AutoModelForCausalLM resolves correct arch.
    # "qwen3_5" → Qwen3_5ForConditionalGeneration (nested weights), wrong for text-only.
    # "qwen3_5_text" → Qwen3_5ForCausalLM (flat weights), correct.
    from transformers import AutoConfig
    model_config = AutoConfig.from_pretrained(model_id)
    if getattr(model_config, "model_type", "") == "qwen3_5":
        model_config.model_type = "qwen3_5_text"
        log.info(f"  [{model_spec['name']}] Pre-patched AutoConfig model_type: qwen3_5 → qwen3_5_text")

    load_kwargs = dict(pretrained_model_name_or_path=model_id, torch_dtype=dtype,
                       config=model_config)

    if is_large:
        # Large models (≥7B): load in 4-bit NF4 to avoid OOM during merge on T4 (16GB VRAM).
        # 9B fp16 = 18GB (OOM), 9B 4-bit = ~5GB disk + ~5GB VRAM → fits comfortably.
        log.info(f"  [{model_spec['name']}] Large model ({model_spec['param_count']}B) — using 4-bit NF4 merge ...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_quant_type       = "nf4",
            bnb_4bit_compute_dtype    = dtype,
            bnb_4bit_use_double_quant = True,
        )
        load_kwargs["quantization_config"] = bnb_config
        load_kwargs["device_map"]          = "auto"   # 4-bit quantization requires GPU
    else:
        # Small models (4B): CPU merge — zero VRAM used during merge.
        log.info(f"  [{model_spec['name']}] Small model ({model_spec['param_count']}B) — CPU merge ...")
        load_kwargs["device_map"] = "cpu"

    if model_class == "causal_lm":
        base_model = AutoModelForCausalLM.from_pretrained(**load_kwargs)
    else:
        base_model = AutoModelForImageTextToText.from_pretrained(**load_kwargs)

    log.info(f"  [{model_spec['name']}] Merging LoRA adapter from {adapter_path} ...")
    peft_model   = PeftModel.from_pretrained(base_model, str(adapter_path))
    merged_model = peft_model.merge_and_unload()

    merged_path.mkdir(parents=True, exist_ok=True)
    merged_model.save_pretrained(str(merged_path), safe_serialization=True)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.save_pretrained(str(merged_path))

    del base_model, peft_model, merged_model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Patch saved config.json for Qwen3.5 only: "qwen3_5" → "qwen3_5_text".
    # Scoped to Qwen3.5 — do NOT patch other models (e.g. Gemma4).
    cfg_path = merged_path / "config.json"
    with open(cfg_path) as f:
        model_cfg = json.load(f)
    current_type = model_cfg.get("model_type", "")
    if current_type == "qwen3_5":
        model_cfg["model_type"] = "qwen3_5_text"
        with open(cfg_path, "w") as f:
            json.dump(model_cfg, f, indent=2)
        log.info(f"  [{model_spec['name']}] Patched saved config model_type: qwen3_5 → qwen3_5_text")

    log.info(f"  [{model_spec['name']}] Merge complete → {merged_path}")
    return merged_path

print("✓ Section 2: merge_adapter_to_disk defined")

## Section 3 — Prompt Builder

Applies the model's chat template to produce the raw prompt string for vLLM.
Must match the format used in Phase 2 training exactly — same system prompt,
same user message structure, same `/no_think` suffix.

In [ ]:
def build_chat_prompt(feature_text: str, pn_history: str, tokenizer) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"Note: \"{pn_history.strip()}\"\n"
            f"Feature: {feature_text}\n\n"
            "/no_think"
        )},
    ]
    # enable_thinking=False suppresses <think> blocks for Qwen3.5 models.
    # Without this, model outputs <think>...</think> + JSON, consuming tokens
    # and causing 300-token limit to cut off the actual JSON answer.
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("✓ Section 3: build_chat_prompt defined")

## Section 4 — vLLM Engine Lifecycle

`init_engine`: loads a merged model into vLLM with `enforce_eager=True`.

For the 9B model, the merged checkpoint was saved in 4-bit NF4 — vLLM auto-detects the
quantization config from the saved `config.json` and loads accordingly (~5 GB VRAM vs 18 GB fp16).

`destroy_engine`: fully frees GPU memory between models. The order matters:
1. `destroy_model_parallel()` — clears distributed process groups
2. `torch.distributed.destroy_process_group()` — clears distributed state
3. `del llm` → `gc.collect()` → `cuda.empty_cache()` → `cuda.synchronize()`

Without `enforce_eager=True`, compiled CUDA graphs (~3.4 GB) remain resident
after `del llm`, causing OOM when the next model loads (vLLM issue #36973).

In [ ]:
def init_engine(merged_path: Path, model_spec: dict, cfg: dict) -> LLM:
    log.info(f"  [{model_spec['name']}] Initialising vLLM engine ...")
    llm = LLM(
        model                  = str(merged_path),
        dtype                  = model_spec["vllm_dtype"],
        gpu_memory_utilization = cfg["GPU_MEM_UTIL"],
        max_model_len          = cfg["MAX_MODEL_LEN"],
        enforce_eager          = cfg["ENFORCE_EAGER"],
        trust_remote_code      = False,
        seed                   = cfg["SEED"],
    )
    log.info(f"  [{model_spec['name']}] vLLM engine ready.")
    return llm


def destroy_engine(llm: LLM, model_name: str) -> None:
    log.info(f"  [{model_name}] Destroying vLLM engine ...")
    destroy_model_parallel()
    with contextlib.suppress(Exception):
        torch.distributed.destroy_process_group()
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()
        free_gb  = torch.cuda.mem_get_info()[0] / 1024**3
        total_gb = torch.cuda.mem_get_info()[1] / 1024**3
        log.info(f"  [{model_name}] VRAM after cleanup: {free_gb:.1f}/{total_gb:.1f} GB")

print("✓ Section 4: init_engine, destroy_engine defined")

## Section 5 — Inference Runner (One Model at a Time)

For each test row:
1. Build the chat prompt string
2. Build a per-note regex FSM (Section 1) as the guided decoding constraint
3. Pass all `(prompt, SamplingParams)` pairs to `llm.generate()` in one call

Each row gets its **own** `SamplingParams` with a unique regex — vLLM accepts
a list of `SamplingParams` for heterogeneous constrained decoding.

In [ ]:
def _parse_json_output(raw_text: str) -> list:
    raw_text = re.sub(r'<think>.*?</think>', '', raw_text, flags=re.DOTALL).strip()
    try:
        parsed = json.loads(raw_text)
        return [s.strip() for s in parsed.get("spans", []) if isinstance(s, str) and s.strip()]
    except (json.JSONDecodeError, AttributeError):
        match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group())
                return [s.strip() for s in parsed.get("spans", []) if isinstance(s, str) and s.strip()]
            except json.JSONDecodeError:
                pass
        return []


def run_inference_for_model(llm, test_rows, pn_map, feat_map, tokenizer, cfg, model_name) -> list:
    log.info(f"  [{model_name}] Building prompts and per-note regex constraints ...")
    prompts, params_list = [], []

    for _, row in test_rows.iterrows():
        pn_history   = pn_map.get(row["pn_num"], "").replace("\n", " ").strip()
        feature_text = feat_map.get((row["case_num"], row["feature_num"]), "")
        prompts.append(build_chat_prompt(feature_text, pn_history, tokenizer))
        regex  = build_constraint_regex(pn_history, cfg["MAX_SPANS_PER_FEATURE"])
        if GuidedDecodingParams is not None:
            guided = GuidedDecodingParams(regex=regex, backend="xgrammar")
            sp = SamplingParams(
                temperature=cfg["LLM_TEMPERATURE"], max_tokens=cfg["MAX_NEW_TOKENS"],
                guided_decoding=guided,
            )
        else:
            sp = SamplingParams(temperature=cfg["LLM_TEMPERATURE"], max_tokens=cfg["MAX_NEW_TOKENS"])
        params_list.append(sp)

    log.info(f"  [{model_name}] Running inference on {len(prompts)} rows ...")
    outputs   = llm.generate(prompts=prompts, sampling_params=params_list)

    all_spans = []
    for output in tqdm(outputs, desc=f"  [{model_name}] Parsing", leave=False):
        raw_text = output.outputs[0].text.strip() if output.outputs else ""
        all_spans.append(_parse_json_output(raw_text))

    n_nonempty = sum(1 for s in all_spans if s)
    log.info(f"  [{model_name}] Done — non-empty: {n_nonempty}/{len(all_spans)}")
    return all_spans

print("✓ Section 5: _parse_json_output, run_inference_for_model defined")

## Section 5b — Transformers Fallback Inference

Used when `USE_VLLM=False` — e.g. if vLLM routes Qwen3.5/Gemma4 to vision-language
handler (vLLM ≤0.19.1 bug) causing `preprocessor_config.json` errors.

Slower than vLLM (row-by-row vs batched), but works on any transformers-compatible model.

In [ ]:
def run_inference_transformers(merged_path: Path, test_rows, pn_map, feat_map,
                               tokenizer, cfg, model_name, model_spec: dict = None) -> list:
    """Transformers generate() fallback — no vLLM required."""
    from transformers import AutoConfig, AutoModelForCausalLM, AutoModelForImageTextToText
    log.info(f"  [{model_name}] Loading merged model for transformers inference ...")

    model_class = (model_spec or {}).get("model_class", "causal_lm")
    load_kwargs  = dict(torch_dtype=torch.bfloat16, device_map="auto")

    # For causal_lm: patch model_type if qwen3_5 so correct arch is resolved
    if model_class == "causal_lm":
        saved_config = AutoConfig.from_pretrained(str(merged_path))
        if getattr(saved_config, "model_type", "") == "qwen3_5":
            saved_config.model_type = "qwen3_5_text"
            load_kwargs["config"] = saved_config
        model = AutoModelForCausalLM.from_pretrained(str(merged_path), **load_kwargs)
    else:
        model = AutoModelForImageTextToText.from_pretrained(str(merged_path), **load_kwargs)

    model.eval()
    log.info(f"  [{model_name}] Running transformers inference on {len(test_rows)} rows ...")
    all_spans = []
    for _, row in tqdm(test_rows.iterrows(), total=len(test_rows), desc=f"  [{model_name}]"):
        pn_history   = pn_map.get(row["pn_num"], "").replace("\n", " ").strip()
        feature_text = feat_map.get((row["case_num"], row["feature_num"]), "")
        prompt  = build_chat_prompt(feature_text, pn_history, tokenizer)
        inputs  = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=cfg["MAX_NEW_TOKENS"],
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_ids  = out_ids[0, inputs["input_ids"].shape[1]:]
        raw_text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
        all_spans.append(_parse_json_output(raw_text))
    n_nonempty = sum(1 for s in all_spans if s)
    log.info(f"  [{model_name}] Done — non-empty: {n_nonempty}/{len(all_spans)}")
    del model; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()
    return all_spans

print("✓ Section 5b: run_inference_transformers defined")

## Section 6 — Character-Level Majority Voting

For each test row:
1. Each model's predicted spans are mapped to `(start, end)` offsets → binary character array
2. The 3 arrays are summed element-wise
3. Characters with vote count ≥ `VOTE_THRESHOLD` (default 2) are accepted
4. Contiguous accepted runs form the final span list

This is more robust than span-level voting: models may agree on the region but differ
slightly on exact boundaries — character voting handles that gracefully.

In [ ]:
def spans_to_char_array(span_locations: list, note_len: int) -> np.ndarray:
    arr = np.zeros(note_len, dtype=np.uint8)
    for start, end in span_locations:
        arr[max(0, start):min(note_len, end)] = 1
    return arr


def char_array_to_spans(arr: np.ndarray) -> list:
    spans, n, i = [], len(arr), 0
    while i < n:
        if arr[i] == 1:
            start = i
            while i < n and arr[i] == 1: i += 1
            spans.append((start, i))
        else:
            i += 1
    return spans


def locate_span_in_note(span_text: str, pn_history: str, score_cutoff: float = 70.0) -> Optional[tuple]:
    span_text = span_text.strip()
    if not span_text or not pn_history:
        return None
    idx = pn_history.find(span_text)
    if idx != -1: return (idx, idx + len(span_text))
    idx = pn_history.lower().find(span_text.lower())
    if idx != -1: return (idx, idx + len(span_text))
    result = partial_ratio_alignment(span_text, pn_history, score_cutoff=score_cutoff)
    if result is not None: return (result.dest_start, result.dest_end)
    return None


def character_level_majority_vote(model_predictions, test_rows, pn_map, vote_threshold=2, fuzzy_cutoff=70.0) -> list:
    n_models, n_rows = len(model_predictions), len(test_rows)
    log.info(f"Majority vote ({n_models} models, threshold={vote_threshold}/{n_models}) ...")
    final_spans = []

    for seq_idx, (_, row) in enumerate(tqdm(test_rows.iterrows(), total=n_rows, desc="Majority vote")):
        pn_history = pn_map.get(row["pn_num"], "")
        note_len   = len(pn_history)
        if note_len == 0:
            final_spans.append([]); continue

        vote_array = np.zeros(note_len, dtype=np.int8)
        for model_idx in range(n_models):
            locations = [loc for text in model_predictions[model_idx][seq_idx]
                         if (loc := locate_span_in_note(text, pn_history, fuzzy_cutoff)) is not None]
            if locations:
                vote_array += spans_to_char_array(locations, note_len)

        consensus = (vote_array >= vote_threshold).astype(np.uint8)
        for i, ch in enumerate(pn_history):
            if ch in (' ', '\t', '\n', '\r') and consensus[i]:
                is_start = (i == 0 or consensus[i-1] == 0)
                is_end   = (i == note_len-1 or consensus[i+1] == 0)
                if is_start or is_end: consensus[i] = 0

        final_spans.append(char_array_to_spans(consensus))

    non_empty = sum(1 for s in final_spans if s)
    log.info(f"Vote complete — non-empty: {non_empty}/{n_rows}")
    return final_spans

print("✓ Section 6: spans_to_char_array, char_array_to_spans, locate_span_in_note, character_level_majority_vote defined")

## Section 7 — Submission Formatter

Converts `(start, end)` span tuples to Kaggle's format: `"10 25;40 55"` (semicolon-separated).

Also strips leading/trailing whitespace from span boundaries and merges overlapping/adjacent
spans that may result from the character-level voting.

In [ ]:
def format_location_string(spans: list, pn_history: str) -> str:
    if not spans: return ""
    clean = []
    for start, end in sorted(spans):
        while start < end and pn_history[start] in (' ', '\t', '\n', '\r'): start += 1
        while end > start and pn_history[end-1] in (' ', '\t', '\n', '\r'): end -= 1
        if start < end: clean.append((start, end))

    merged = []
    for start, end in sorted(clean):
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))

    return ";".join(f"{s} {e}" for s, e in merged) if merged else ""


def build_submission(final_spans: list, test_df: pd.DataFrame, pn_map: dict) -> pd.DataFrame:
    rows = []
    for row_idx, (_, test_row) in enumerate(test_df.iterrows()):
        pn_history = pn_map.get(test_row["pn_num"], "")
        spans      = final_spans[row_idx] if row_idx < len(final_spans) else []
        location   = format_location_string(spans, pn_history)
        rows.append({"id": test_row["id"], "location": location if location else np.nan})
    return pd.DataFrame(rows)

print("✓ Section 7: format_location_string, build_submission defined")

## Run Phase 3 — Generate Submission

For each model:
1. Merge LoRA adapter into base weights on CPU (no VRAM used)
2. Load merged model into vLLM
3. Run inference on all test rows
4. Destroy vLLM engine + delete merged model from disk

Then run character-level majority voting and save `submission.csv`.

Expected log output per model:
```
Processing model: qwen_35_4b
  Merging LoRA adapter ...  ✓
  Initialising vLLM engine ...  ready
  Running inference on N rows ...
  Done — non-empty: M/N
  VRAM after cleanup: X.X/Y.Y GB
```

In [ ]:
def main():
    cfg = CONFIG
    print("\n" + "="*65)
    print("  PHASE 3: Kaggle Inference")
    print(f"  Mode: {'vLLM (auto-fallback to transformers)' if cfg['USE_VLLM'] else 'transformers'}")
    print("="*65 + "\n")

    print("▶ Loading test data ...")
    data_dir = cfg["DATA_DIR"]
    test_df  = pd.read_csv(data_dir / "test.csv")
    pn_df    = pd.read_csv(data_dir / "patient_notes.csv")
    feat_df  = pd.read_csv(data_dir / "features.csv")
    pn_map   = pn_df.set_index("pn_num")["pn_history"].to_dict()
    feat_map = feat_df.set_index(["case_num", "feature_num"])["feature_text"].to_dict()
    print(f"  Test rows: {len(test_df)}")

    tmp_root = cfg["OUTPUT_DIR"] / "merged_models"
    tmp_root.mkdir(parents=True, exist_ok=True)

    all_model_predictions = []
    for i, model_spec in enumerate(MODEL_REGISTRY):
        model_name = model_spec["name"]
        print(f"\n{'='*65}")
        print(f"  Model {i+1}/{len(MODEL_REGISTRY)}: {model_name}")
        print(f"{'='*65}")

        merged_path = merge_adapter_to_disk(model_spec, tmp_root, cfg)

        tokenizer = AutoTokenizer.from_pretrained(str(merged_path))
        if tokenizer.pad_token is None:
            tokenizer.pad_token    = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id

        if cfg["USE_VLLM"]:
            try:
                llm         = init_engine(merged_path, model_spec, cfg)
                model_spans = run_inference_for_model(llm, test_df, pn_map, feat_map, tokenizer, cfg, model_name)
                all_model_predictions.append(model_spans)
                destroy_engine(llm, model_name)
                del llm
            except Exception as e:
                log.warning(f"  [{model_name}] vLLM failed ({e}) — falling back to transformers")
                with contextlib.suppress(Exception):
                    destroy_model_parallel()
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache(); torch.cuda.synchronize()
                model_spans = run_inference_transformers(merged_path, test_df, pn_map, feat_map, tokenizer, cfg, model_name, model_spec)
                all_model_predictions.append(model_spans)
        else:
            model_spans = run_inference_transformers(merged_path, test_df, pn_map, feat_map, tokenizer, cfg, model_name, model_spec)
            all_model_predictions.append(model_spans)

        del tokenizer; gc.collect()
        shutil.rmtree(str(merged_path), ignore_errors=True)
        print(f"  ✓ [{model_name}] merged model deleted from disk")

    print("\n▶ Running character-level majority vote ...")
    n_models_active     = len(all_model_predictions)
    effective_threshold = min(cfg["VOTE_THRESHOLD"], n_models_active)
    final_spans = character_level_majority_vote(
        all_model_predictions, test_df, pn_map,
        vote_threshold=effective_threshold, fuzzy_cutoff=cfg["FUZZY_SCORE_CUTOFF"],
    )

    submission_df = build_submission(final_spans, test_df, pn_map)
    out_path = cfg["OUTPUT_DIR"] / "submission.csv"
    submission_df.to_csv(out_path, index=False)

    print("\n" + "="*65)
    print(f"  ✓ Submission saved → {out_path}")
    print(f"  Shape: {submission_df.shape}")
    print(f"  Non-empty: {submission_df['location'].notna().sum()} / {len(submission_df)}")
    print("="*65)
    print(submission_df.head(10).to_string())

main()